# Making final dataset before tableau stuff begins

# Setup

In [1]:
# imports
import pandas as pd
from idx.components import ingest, preprocessing, feature_engineering, cleaning
from sklearn.pipeline import Pipeline
import geopandas as gpd

In [2]:
# making school district geodataframe
district_gdf = gpd.read_file("../data/raw/district/DistrictAreas2425.shp")
district_gdf = district_gdf.to_crs(epsg=4326)

# initializing pipeline components
ingestor = ingest.MLSIngestor(input_path="../data/raw/")
fred_ingestor = ingest.FredMerger()
cleaner = preprocessing.DataCleaner()
flagger = preprocessing.BadDataFlagger()
market_maker = feature_engineering.CreateMarketMetrics()
cleanup = feature_engineering.CleanUpTransformer()
null = feature_engineering.NullDropper()
dist_merger = feature_engineering.DistrictMerger(district_gdf=district_gdf)
iqr_flagger = cleaning.OutlierFlagger(subset=["OriginalListPrice", "LivingArea", "LotSizeArea"])

ingest_test = Pipeline([ 
    ("ingestor", ingestor),
    ("fred_ingestor", fred_ingestor),
    ("cleaner", cleaner),
    ("flagger", flagger),
    ("market_maker", market_maker),
    ("cleanup", cleanup),
    ("null", null),
    ("dist_merger", dist_merger),
    ("iqr_flagger", iqr_flagger)
])


df_sold, df_listings = ingest_test.fit_transform(X=None, y=None)

Read 31 files from ../data/raw with filter 'Sold'


Read 28 files from ../data/raw with filter 'Listing'


Merged FRED data with sold data: 681599 rows
Merged FRED data with listings data: 893594 rows


Original sold_df shape: (681599, 87)
Original listings_df shape: (893594, 87)
Post-drop sold_df shape: (681599, 72)
Post-drop listings_df shape: (893594, 61)
4 datetime columns converted and 5 integer columns converted.
Dropped 9 non-analysis columns from sold_df and listings_df.
Flagged 215 rows with impossible measurements.
Flagged 0 rows with impossible year.
Flagged 124 rows with listing after close.
Flagged 435 rows with purchase after close.
Flagged 430 rows with negative timeline.
Flagged 19411 rows with null coordinates.
Flagged 46 rows with placeholder coordinates.
Flagged 232 rows with non-California coordinates.
Flagged 223 rows with impossible measurements.
Flagged 41 rows with impossible year.
Flagged 140 rows with listing after close.
Flagged 351 rows with purchase after c

In [3]:
display(df_listings.head())

,OriginalListPrice,ListingKey,CloseDate,ClosePrice,Latitude,Longitude,UnparsedAddress,PropertyType,LivingArea,ListPrice,...,non_cali_coords_flag,price_ratio,price_per_sqft,days_on_market,yr_month,listing_to_contract_days,contract_to_close_days,index_right,DistrictNa,outlier_flag
2,1400000.0,1076193814,NaT,NaN,33.858559,-116.542169,505 E Molino Road,Residential,2256.0,1400000.0,...,False,NaN,NaN,NaN,NaT,NaN,NaN,478.0,Palm Springs Unified,False
8,199990.0,1076192982,NaT,NaN,32.777540,-115.553772,1826 S 4th Street,Residential,1024.0,199990.0,...,False,NaN,NaN,NaN,NaT,NaN,NaN,149.0,Central Union High,False
8,199990.0,1076192982,NaT,NaN,32.777540,-115.553772,1826 S 4th Street,Residential,1024.0,199990.0,...,False,NaN,NaN,NaN,NaT,NaN,NaN,150.0,El Centro Elementary,False
11,835000.0,1076190486,NaT,NaN,33.166112,-117.266548,4645 Cordoba Way,Residential,1444.0,835000.0,...,False,NaN,NaN,NaN,NaT,NaN,NaN,582.0,Vista Unified,False
12,1150000.0,1076190201,NaT,NaN,33.212280,-117.218628,1517 Via Pedro,Residential,3309.0,1150000.0,...,False,NaN,NaN,NaN,NaT,NaN,NaN,582.0,Vista Unified,False


In [4]:
display(df_listings.info())

<class 'pandas.core.frame.DataFrame'>
Index: 938118 entries, 2 to 893593
Data columns (total 64 columns):
 #   Column                       Non-Null Count   Dtype         
---  ------                       --------------   -----         
 0   OriginalListPrice            938118 non-null  float64       
 1   ListingKey                   938118 non-null  int64         
 2   CloseDate                    276734 non-null  datetime64[ns]
 3   ClosePrice                   253027 non-null  float64       
 4   Latitude                     938118 non-null  float64       
 5   Longitude                    938118 non-null  float64       
 6   UnparsedAddress              935777 non-null  object        
 7   PropertyType                 938118 non-null  object        
 8   LivingArea                   823521 non-null  float64       
 9   ListPrice                    938096 non-null  float64       
 10  DaysOnMarket                 938118 non-null  int64         
 11  ListOfficeName               93

None

# something

In [5]:
# TODO : save a df with bad data removed and one with all the flags still
# Include a written comparison of dataset size and median values before and after filtering. OK 

In [6]:
def drop_flagged(df, flag_columns):
    """
    Drops rows from the DataFrame where any of the specified flag columns are True.
    
    Parameters:
    df (pd.DataFrame): The input DataFrame.
    flag_columns (list): List of column names that contain boolean flags.
    
    Returns:
    pd.DataFrame: A new DataFrame with flagged rows removed.
    """
    # Create a boolean mask where any of the flag columns are True
    mask = df[flag_columns].any(axis=1)
    
    # Return a new DataFrame with flagged rows removed
    return df[~mask]

In [7]:
flag_columns = [col for col in df_listings.columns if 'flag' in col.lower()]

df_sold_cleared = drop_flagged(df_sold, flag_columns)
df_listings_cleared = drop_flagged(df_listings, flag_columns)


In [8]:
display(df_sold_cleared.info())
display(df_sold_cleared.shape)

<class 'pandas.core.frame.DataFrame'>
Index: 689609 entries, 0 to 681594
Data columns (total 75 columns):
 #   Column                       Non-Null Count   Dtype         
---  ------                       --------------   -----         
 0   Flooring                     402912 non-null  object        
 1   ViewYN                       623498 non-null  object        
 2   WaterfrontYN                 393 non-null     object        
 3   BasementYN                   10304 non-null   object        
 4   PoolPrivateYN                607149 non-null  object        
 5   OriginalListPrice            689609 non-null  float64       
 6   ListingKey                   689609 non-null  int64         
 7   CloseDate                    689609 non-null  datetime64[ns]
 8   ClosePrice                   689609 non-null  float64       
 9   Latitude                     689609 non-null  float64       
 10  Longitude                    689609 non-null  float64       
 11  UnparsedAddress              68

None

(689609, 75)

In [9]:
display(df_listings_cleared.info())
display(df_listings_cleared.shape)

<class 'pandas.core.frame.DataFrame'>
Index: 797240 entries, 2 to 893593
Data columns (total 64 columns):
 #   Column                       Non-Null Count   Dtype         
---  ------                       --------------   -----         
 0   OriginalListPrice            797240 non-null  float64       
 1   ListingKey                   797240 non-null  int64         
 2   CloseDate                    246339 non-null  datetime64[ns]
 3   ClosePrice                   224012 non-null  float64       
 4   Latitude                     797240 non-null  float64       
 5   Longitude                    797240 non-null  float64       
 6   UnparsedAddress              795273 non-null  object        
 7   PropertyType                 797240 non-null  object        
 8   LivingArea                   718951 non-null  float64       
 9   ListPrice                    797221 non-null  float64       
 10  DaysOnMarket                 797240 non-null  int64         
 11  ListOfficeName               79

None

(797240, 64)

In [10]:
print("Original df_sold shape:", df_sold.shape)
print("Cleared df_sold shape:", df_sold_cleared.shape)
print("Original df_listings shape:", df_listings.shape)
print("Cleared df_listings shape:", df_listings_cleared.shape)

Original df_sold shape: (803696, 75)
Cleared df_sold shape: (689609, 75)
Original df_listings shape: (938118, 64)
Cleared df_listings shape: (797240, 64)


In [11]:
df_listings_cleared.head()

,OriginalListPrice,ListingKey,CloseDate,ClosePrice,Latitude,Longitude,UnparsedAddress,PropertyType,LivingArea,ListPrice,...,non_cali_coords_flag,price_ratio,price_per_sqft,days_on_market,yr_month,listing_to_contract_days,contract_to_close_days,index_right,DistrictNa,outlier_flag
2,1400000.0,1076193814,NaT,NaN,33.858559,-116.542169,505 E Molino Road,Residential,2256.0,1400000.0,...,False,NaN,NaN,NaN,NaT,NaN,NaN,478.0,Palm Springs Unified,False
8,199990.0,1076192982,NaT,NaN,32.777540,-115.553772,1826 S 4th Street,Residential,1024.0,199990.0,...,False,NaN,NaN,NaN,NaT,NaN,NaN,149.0,Central Union High,False
8,199990.0,1076192982,NaT,NaN,32.777540,-115.553772,1826 S 4th Street,Residential,1024.0,199990.0,...,False,NaN,NaN,NaN,NaT,NaN,NaN,150.0,El Centro Elementary,False
11,835000.0,1076190486,NaT,NaN,33.166112,-117.266548,4645 Cordoba Way,Residential,1444.0,835000.0,...,False,NaN,NaN,NaN,NaT,NaN,NaN,582.0,Vista Unified,False
12,1150000.0,1076190201,NaT,NaN,33.212280,-117.218628,1517 Via Pedro,Residential,3309.0,1150000.0,...,False,NaN,NaN,NaN,NaT,NaN,NaN,582.0,Vista Unified,False


In [12]:
"""df_sold_cleared.to_csv("../data/processed/df_sold_cleared.csv", index=False)
df_listings_cleared.to_csv("../data/processed/df_listings_cleared.csv", index=False)
df_sold.to_csv("../data/processed/df_sold.csv", index=False)
df_listings.to_csv("../data/processed/df_listings.csv", index=False)"""

'df_sold_cleared.to_csv("../data/processed/df_sold_cleared.csv", index=False)\ndf_listings_cleared.to_csv("../data/processed/df_listings_cleared.csv", index=False)\ndf_sold.to_csv("../data/processed/df_sold.csv", index=False)\ndf_listings.to_csv("../data/processed/df_listings.csv", index=False)'